# Python Decorators — a practical, friendly notebook

This notebook explains **decorators** in Python step-by-step with examples you can run.

**What you'll learn**
- Functions are objects (can be passed around)
- The core decorator idea: `f = decorator(f)`
- Writing a decorator with a wrapper (`*args, **kwargs`)
- Preserving metadata with `functools.wraps`
- Decorators with arguments (decorator factories)
- Stacking decorators and understanding order
- Common pitfalls + best practices


## 1) Functions are objects

In Python, functions can be:
- assigned to variables
- passed as arguments
- returned from other functions


In [3]:
def greet(name):
    return f"Hello, {name}!"

alias = greet  # assign function to another name

print(greet("Ada"))
print(alias("Grace"))  # same function
print("Function name:", greet.__name__)


Hello, Ada!
Hello, Grace!
Function name: greet


### Passing functions as arguments

In [4]:
def call_twice(fn, x):
    return fn(x), fn(x)

def square(n):
    return n * n

print(call_twice(square, 5))


(25, 25)


## 2) The decorator idea: wrap one function with another

A decorator is a callable that takes a function and returns a new function.

In other words:

```python
decorated = decorator(original)
```

Python's `@decorator` syntax is just a nicer way to write that.


In [5]:
def excited(fn):
    # fn is a function passed in
    def wrapper(name):
        return fn(name).upper() + "!!!"
    return wrapper

# Without @ syntax:
excited_greet = excited(greet)

print(greet("Ada"))
print(excited_greet("Ada"))


Hello, Ada!
HELLO, ADA!!!!


### The `@` syntax (syntactic sugar)

This:

```python
@excited
def greet(name): ...
```

means:

```python
def greet(name): ...
greet = excited(greet)
```


In [6]:
@excited
def greet2(name):
    return f"Hello, {name}"

print(greet2("Ada"))


HELLO, ADA!!!


## 3) Writing a reusable decorator with `*args, **kwargs`

Most functions have different signatures, so a general wrapper should accept any arguments:

- `*args` captures positional args
- `**kwargs` captures keyword args


In [7]:
def log_calls(fn):
    def wrapper(*args, **kwargs):
        print(f"[log] Calling {fn.__name__} with args={args}, kwargs={kwargs}")
        result = fn(*args, **kwargs)
        print(f"[log] {fn.__name__} returned {result!r}")
        return result
    return wrapper

@log_calls
def add(a, b):
    return a + b

@log_calls
def power(base, exp=2):
    return base ** exp

print(add(2, 3))
print(power(5))
print(power(5, exp=3))


[log] Calling add with args=(2, 3), kwargs={}
[log] add returned 5
5
[log] Calling power with args=(5,), kwargs={}
[log] power returned 25
25
[log] Calling power with args=(5,), kwargs={'exp': 3}
[log] power returned 125
125


## 4) Why `functools.wraps` matters

A naive decorator replaces the function object, which can lose metadata like:
- `__name__`
- `__doc__`
- signature info in help/tools

Use `functools.wraps(fn)` to preserve it.


In [ ]:
import functools

def log_calls_wrapped(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print(f"[log] Calling {fn.__name__}")
        return fn(*args, **kwargs)
    return wrapper

@log_calls
def example_plain(x):
    """Plain decorator (no wraps)."""
    return x

@log_calls_wrapped
def example_wrapped(x):
    """Wrapped decorator (preserves metadata)."""
    return x

print("example_plain.__name__:", example_plain.__name__)
print("example_wrapped.__name__:", example_wrapped.__name__)
print("example_plain.__doc__:", example_plain.__doc__)
print("example_wrapped.__doc__:", example_wrapped.__doc__)


example_plain.__name__: wrapper
example_wrapped.__name__: example_wrapped
example_plain.__doc__: None
example_wrapped.__doc__: Wrapped decorator (preserves metadata).


Signature: example_wrapped(x)
Source:   
@log_calls_wrapped
def example_wrapped(x):
    """Wrapped decorator (preserves metadata)."""
    return x
File:      /tmp/ipykernel_527487/2880948185.py
Type:      function

## 5) A practical decorator: timing function execution

This is a common real-world use: measure how long a function takes.


In [ ]:
import time
import functools

def time_it(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        start = time.perf_counter()
        result = fn(*args, **kwargs)
        end = time.perf_counter()
        print(f"[time] {fn.__name__} took {(end-start)*1000:.2f} ms")
        return result
    return wrapper

@time_it
def slow_sum(n):
    total = 0
    for i in range(n):
        total += i
    return total

slow_sum(200_000)


[time] slow_sum took 22.12 ms


19999900000

## My Example

In [27]:
def datetime2SMtime(datetime):
    """
    For converting datetime into a string for SuperMAG api functions
    Parameters
    ----------
    datetime : datetime.datetime or pandas.Timestamp

    Returns
    -------
    datetime : string
    """
    return ''.join('T'.join(str(datetime).split(' ')).split(':'))[:-2]
def numpy2SMtime(numpy_time):
    """
    For converting numpy datetimes into a string for SuperMAG apu functions

    Parameters
    ----------
    numpy_time : numpy.datetime64

    Returns
    -------
    datetime : string
        DESCRIPTION.

    """
    return ''.join(np.datetime_as_string(numpy_time, 'm').split(':'))
def check_time(func):
    """
    For checking the time inputs and coverting from python datetimes, pandas timestamps
    or numpy datetimes to a string that the SuperMag api understands

    Parameters
    ----------
    func : definition
        Intitial function that needs time checking.

    Raises
    ------
    ValueError
        If the time argument cannot not be understood /hasn't been implemented.

    Returns
    -------
    output
        Function output.

    """
    @functools.wraps(func)
    def wrapper(time, *args, **kwargs):
        from datetime import datetime as dt
        if isinstance(time, (pd.Timestamp, dt)):
            time= datetime2SMtime(time)
        elif isinstance(time, np.datetime64):
            time= numpy2SMtime(time)
        elif not isinstance(time, str):
            raise ValueError('Time format not understood')
        return func(time, *args, **kwargs)
    return wrapper

@check_time
def func(time, x, y, z=False):
    print(f"time type={type(time)}, time={time}, x={x}, y={y}, z={z}")
import numpy as np
import pandas as pd
func(np.datetime64('2024-01-01T12:00:00'), x=10, y=20)
func(pd.Timestamp('2024-01-01T12:00:00'), 10, 20, z=True)


time type=<class 'str'>, time=2024-01-01T1200, x=10, y=20, z=False
time type=<class 'str'>, time=2024-01-01T1200, x=10, y=20, z=True


## 6) Decorators with arguments (decorator factories)

Sometimes you want to configure a decorator:

```python
@repeat(times=3)
def f(...):
    ...
```

To do this, you write a function that **returns** a decorator.


In [10]:
import functools

def repeat(times=2):
    # This outer function captures configuration (times)
    def decorator(fn):
        @functools.wraps(fn)
        def wrapper(*args, **kwargs):
            last = None
            for _ in range(times):
                last = fn(*args, **kwargs)
            return last
        return wrapper
    return decorator

@repeat(times=3)
def ping():
    print("ping")
    return "done"

print("Result:", ping())


ping
ping
ping
Result: done


## My Example

In [21]:
import inspect
import functools

def enforce_types(**type_map):
    """
    Lightweight runtime argument type checking.

    Usage:
        @enforce_types(path=str, site_code=str, opts=(dict, type(None)))
        def func(path, site_code, opts=None, **kwargs):
            ...

    Each key is the parameter name; each value is either a single type
    or a tuple of allowed types.
    """
    def decorator(func):
        sig = inspect.signature(func)

        @functools.wraps(func)
        def wrapper(*args, **kwargs):
            bound = sig.bind_partial(*args, **kwargs)
            bound.apply_defaults()

            for name, expected in type_map.items():
                if name not in bound.arguments:
                    continue

                value = bound.arguments[name]
                # Allow None when explicitly included via type(None)
                if isinstance(expected, tuple):
                    ok = isinstance(value, expected)
                    expected_names = ", ".join(t.__name__ for t in expected)
                else:
                    ok = isinstance(value, expected)
                    expected_names = expected.__name__

                if not ok:
                    raise TypeError(
                        f"Argument '{name}' to {func.__name__}() must be instance of "
                        f"{expected_names}, got {type(value).__name__}"
                    )

            return func(*args, **kwargs)

        return wrapper

    return decorator

@enforce_types(
    inputstr=str,
    positive_answer=str,
    negative_answer=str,
)
def validinput(inputstr, positive_answer, negative_answer):
    """
    Ask for a yes/no style response and enforce one of two allowed answers.

    Parameters
    ----------
    inputstr : str
        Prompt shown to the user.
    positive_answer : str
        Accepted value that maps to True.
    negative_answer : str
        Accepted value that maps to False.

    Returns
    -------
    bool
        True for ``positive_answer``, False for ``negative_answer``.

    Examples
    --------
    >>> validinput('Continue?', 'y', 'n')  # doctest: +SKIP
    True
    """
    answer= input(inputstr+'\n')
    if answer==positive_answer:
        return True
    elif answer== negative_answer:
        return False
    else:
        print('Invalid response should be either '+ str(positive_answer)+ ' or ' +str(negative_answer))
        return validinput(inputstr, positive_answer, negative_answer)
@enforce_types(x=bool, y=(str, type(None)))
def example(x, y=None):
    print(f"x={x}, y={y}")

# print(validinput('Continue?', 12, 'n'))
example(1, y=20)

TypeError: Argument 'x' to example() must be instance of bool, got int

## 7) Stacking decorators and order

When you write:

```python
@A
@B
def f(...):
    ...
```

It becomes:

```python
f = A(B(f))
```

So **B** runs closer to the original function.


In [11]:
import functools

def deco_A(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print("A: before")
        out = fn(*args, **kwargs)
        print("A: after")
        return out
    return wrapper

def deco_B(fn):
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        print("B: before")
        out = fn(*args, **kwargs)
        print("B: after")
        return out
    return wrapper

@deco_A
@deco_B
def hello():
    print("hello!")

hello()


A: before
B: before
hello!
B: after
A: after


## 8) Common pitfalls + best practices

**Pitfalls**
- Forgetting `return result` in wrapper (your function returns `None`)
- Not using `*args, **kwargs` (breaks functions with different signatures)
- Not using `functools.wraps` (metadata lost)

**Best practices**
- Always use `@functools.wraps(fn)` in decorators
- Keep wrappers small and focused
- Prefer composition (stack decorators) over mega-decorators


## 9) Mini-exercises (optional)

Try these:
1. Modify `time_it` to also print the arguments.
2. Write `@cache_results` using a dictionary inside the decorator.
3. Write `@validate_types` that checks types of args (simple version).


In [25]:
# Exercise starter: simple caching decorator

import functools
import time

def cache_results(fn):
    cache = {}
    @functools.wraps(fn)
    def wrapper(*args, **kwargs):
        key = (args, tuple(sorted(kwargs.items())))
        if key in cache:
            print("[cache] hit")
            return cache[key]
        print("[cache] miss")
        result = fn(*args, **kwargs)
        cache[key] = result
        return result
    return wrapper

@time_it
@cache_results
def fib(n):
    time.sleep(0.1)  # Simulate expensive computation
    return n**2
for i in range(10):
     print(i, fib(i))  # warm up cache for fib(10)
for i in range(10):
    print(i, fib(i))  # should be fast second time

[cache] miss
[time] fib took 100.42 ms
0 0
[cache] miss
[time] fib took 100.20 ms
1 1
[cache] miss
[time] fib took 100.29 ms
2 4
[cache] miss
[time] fib took 100.35 ms
3 9
[cache] miss
[time] fib took 100.27 ms
4 16
[cache] miss
[time] fib took 100.26 ms
5 25
[cache] miss
[time] fib took 100.19 ms
6 36
[cache] miss
[time] fib took 101.63 ms
7 49
[cache] miss
[time] fib took 100.35 ms
8 64
[cache] miss
[time] fib took 100.21 ms
9 81
[cache] hit
[time] fib took 0.03 ms
0 0
[cache] hit
[time] fib took 0.01 ms
1 1
[cache] hit
[time] fib took 0.02 ms
2 4
[cache] hit
[time] fib took 0.02 ms
3 9
[cache] hit
[time] fib took 0.02 ms
4 16
[cache] hit
[time] fib took 0.02 ms
5 25
[cache] hit
[time] fib took 0.01 ms
6 36
[cache] hit
[time] fib took 0.01 ms
7 49
[cache] hit
[time] fib took 0.02 ms
8 64
[cache] hit
[time] fib took 0.01 ms
9 81
